# URLinspector 🔬

> "I will look at *this* URL, and I will tell you about it."

Welcome to your first targeted strike. We are going to look at a single page, pull it into our local reality, and expose the **JavaScript Gap**—the massive blind spot that cheap cloud scrapers have because they don't actually render the page.

In [ ]:
# Allows adjusting secret sauce recipe adjustments
%load_ext autoreload
%autoreload 2

In [ ]:
from pipulate import wand
from imports import url_inspect_sauce as sauce
import nest_asyncio
import ipywidgets as widgets
from IPython.display import display
nest_asyncio.apply()

job = "urlinspector-01" 
wand.speak("Wand initialized. Enter your target URL below.")

url_input = widgets.Text(
    value='https://example.com/',
    placeholder='https://your-target-url.com',
    description='🎯 Target:',
    disabled=False,
    layout=widgets.Layout(width='80%')
)
display(url_input)
wand.speak("You can change it from the example.")

wand.imperio()

In [ ]:
# 1. Capture State & Scrape
target_url = url_input.value
wand.set(job, 'target_url', target_url)

wand.speak(
    f"Initializing browser automation for {target_url}. "
    "\nThe browser is going to pop up and just sit there for about eight seconds. This is intentional. "
    "\nWe are waiting out an invisible CAPTCHA to prove to the server that you are a carbon-based lifeform."
    "\nInitializing browser optics. Hands off the mouse! "
)

# The sauce file looks inside the wand's memory for the target
extracted_data = await sauce.scrape(
    job, 
    headless=False, 
    override_cache=False,
    delay_range=None
)

# Since this is a 1-to-1 inspector, we check the first returned item
if extracted_data and extracted_data[0].get('success'):
    if extracted_data[0].get('cached'):
        wand.speak("Cache Hit! Using existing artifacts. If you want to see the browser pop up again, change override_cache to True.")
    else:
        wand.speak("Fresh Scrape Successful.")
        
    # 2. The Optics
    wand.speak("Shattering the DOM into LLM Optics...")
    await sauce.generate_extractions_post_scrape(job, verbose=True)

    wand.show_llm_optics(target_url)

    wand.speak(
        "\nTry clicking dom_hierarchy.html or dom_layout_boxes.html. "
        "\nCompare to the text versions. See a difference? "
        "\nBoth you and the LLM can examine any of these LLM Optics files — artifacts of the scrape. "
    )
else:
    wand.speak("I encountered an error during navigation.")
    error_msg = extracted_data[0].get('error') if extracted_data else 'Unknown error'
    print(f"Scrape Failed: {error_msg}")

wand.imperio()

### 🔍 Exposing the JavaScript Gap
Cheap scrapers only see what the server initially sends (`source.html`). Modern browsers execute JavaScript to build the final page (`rendered_dom.html`). 

Let's calculate the exact structural difference between the two to see how much content is hidden behind JavaScript execution.

In [ ]:
# Calculate the JS Gap
gap_data = sauce.calculate_js_gap(job)

if "error" in gap_data:
    wand.speak("I encountered an error calculating the gap.")
    print(f"Error: {gap_data['error']}")
else:
    wand.speak(
        f"Gap analysis complete. The server sent {gap_data['source_tag_count']} HTML tags. "
        f"After JavaScript execution, there are {gap_data['rendered_tag_count']} tags."
    )
    
    import json
    print("Forensic Delta:")
    print(json.dumps(gap_data, indent=2))
    
    if gap_data['is_js_heavy']:
        print("\n⚠️ WARNING: This page relies heavily on client-side rendering. Legacy crawlers will likely fail to parse its content.")

wand.imperio()

In [ ]:
from pipulate import wand
wand.nbup("Advanced_Notebooks/01_URLinspector", modules=("url_inspect_sauce",))